In [ ]:
import os
import json
import shutil
import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Flatten, Dense
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

SEED = 202                      # separate from split (4) and augmentation (101)
tf.keras.utils.set_random_seed(SEED)

DATA = "/home/admins/rebuild_workspace/06_dataset_final"
train_path = os.path.join(DATA, "Train")
valid_path = os.path.join(DATA, "Validation")
test_path  = os.path.join(DATA, "Test")

word_classes = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']

IDG = ImageDataGenerator(rescale=1/255)

train = IDG.flow_from_directory(train_path, target_size=(224,224),
                                classes=word_classes, batch_size=16,
                                class_mode='categorical')
valid = IDG.flow_from_directory(valid_path, target_size=(224,224),
                                classes=word_classes, batch_size=16,
                                class_mode='categorical', shuffle=False)
test  = IDG.flow_from_directory(test_path, target_size=(224,224),
                                classes=word_classes, batch_size=16,
                                class_mode='categorical', shuffle=False)

print(train.class_indices)

Found 900 images belonging to 10 classes.


Found 200 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
{'bat': 0, 'cup': 1, 'drop': 2, 'eat': 3, 'fish': 4, 'hot': 5, 'jump': 6, 'milk': 7, 'pen': 8, 'red': 9}


In [4]:
model = Sequential([
    Input(shape=(224, 224, 3)),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPool2D((2,2), strides=2),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPool2D((2,2), strides=2),
    Flatten(),
    Dense(256, activation='tanh'),
    Dense(128, activation='tanh'),
    Dense(10,  activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

I0000 00:00:1787298052.434944   19282 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5658 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050, pci bus id: 0000:08:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 200704)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    51,380,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,434,058 (196.21 MB)

 Trainable params: 51,434,058 (196.21 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(x=train, validation_data=valid, epochs=15, verbose=2)

OUT = "/home/admins/rebuild_workspace"
model.save(os.path.join(OUT, "model_rebuilt_data.h5"))

pd.DataFrame(history.history).to_csv(
    os.path.join(OUT, "logs", "training_history_log.csv"), index=False)

with open(os.path.join(OUT, "logs", "class_labels_rebuilt.json"), "w") as f:
    json.dump({v: k for k, v in train.class_indices.items()}, f)

print("\nfinal train acc:", history.history['accuracy'][-1])
print("final val acc  :", history.history['val_accuracy'][-1])

Epoch 1/15


/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1787298243.921825   23119 service.cc:148] XLA service 0x7bc32c004180 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787298243.921870   23119 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2026-08-21 09:44:03.953360: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787298244.086894   23119 cuda_dnn.cc:529] Loaded cuDNN version 92000
I0000 00:00:1787298248.744013   2

57/57 - 13s - 234ms/step - accuracy: 0.0900 - loss: 2.7623 - val_accuracy: 0.1350 - val_loss: 2.3515
Epoch 2/15
57/57 - 2s - 41ms/step - accuracy: 0.1333 - loss: 2.2954 - val_accuracy: 0.1900 - val_loss: 2.2279
Epoch 3/15
57/57 - 2s - 40ms/step - accuracy: 0.3889 - loss: 1.7471 - val_accuracy: 0.4450 - val_loss: 1.6124
Epoch 4/15
57/57 - 2s - 40ms/step - accuracy: 0.8211 - loss: 0.7315 - val_accuracy: 0.5600 - val_loss: 1.2407
Epoch 5/15
57/57 - 2s - 41ms/step - accuracy: 0.9356 - loss: 0.3346 - val_accuracy: 0.5800 - val_loss: 1.0905
Epoch 6/15
57/57 - 2s - 42ms/step - accuracy: 0.9856 - loss: 0.1387 - val_accuracy: 0.6550 - val_loss: 1.0049
Epoch 7/15
57/57 - 2s - 41ms/step - accuracy: 0.9989 - loss: 0.0800 - val_accuracy: 0.6850 - val_loss: 0.9641
Epoch 8/15
57/57 - 2s - 41ms/step - accuracy: 0.9989 - loss: 0.0438 - val_accuracy: 0.6800 - val_loss: 0.9794
Epoch 9/15
57/57 - 2s - 41ms/step - accuracy: 1.0000 - loss: 0.0283 - val_accuracy: 0.7000 - val_loss: 0.9165
Epoch 10/15
57/57 -


final train acc: 1.0
final val acc  : 0.7350000143051147


In [6]:
loss, acc = model.evaluate(test, verbose=1)
print(f"\nFull test (20/class): loss {loss:.4f}  accuracy {acc:.4f}  ({acc*100:.2f}%)")

/home/admins/miniconda3/envs/the_sheff/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.7115 - loss: 0.8467

Full test (20/class): loss 0.8225  accuracy 0.7150  (71.50%)


In [7]:
REAL_DIR = "/home/admins/rebuild_workspace/06_test_real_only"

for word in word_classes:
    dst = os.path.join(REAL_DIR, word)
    os.makedirs(dst, exist_ok=True)
    src = os.path.join(test_path, word)
    for i in range(1, 10):
        fn = f"{i:03d}.png"
        shutil.copy2(os.path.join(src, fn), os.path.join(dst, fn))

test_real = IDG.flow_from_directory(REAL_DIR, target_size=(224,224),
                                    classes=word_classes, batch_size=16,
                                    class_mode='categorical', shuffle=False)

loss_r, acc_r = model.evaluate(test_real, verbose=1)
print(f"\nReal only (9/class): loss {loss_r:.4f}  accuracy {acc_r:.4f}  ({acc_r*100:.2f}%)")

Found 90 images belonging to 10 classes.
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 311ms/step - accuracy: 0.7611 - loss: 0.6393

Real only (9/class): loss 0.6907  accuracy 0.7556  (75.56%)


In [10]:
def report(gen, label):
    gen.reset()
    prob = model.predict(gen, verbose=0)
    pred = np.argmax(prob, axis=1)
    true = gen.classes
    print(f"=== {label} ===")
    print("alignment check (should match evaluate):", (pred == true).mean())
    cm = pd.DataFrame(confusion_matrix(true, pred),
                      index=word_classes, columns=word_classes)
    cm.index.name, cm.columns.name = "Actual", "Predicted"
    print(cm)
    print()
    print(classification_report(true, pred, target_names=word_classes,
                                digits=4, zero_division=0))
    return true, pred, prob, cm

t_full, p_full, pr_full, cm_full = report(test, "FULL TEST (20/class)")
print("\n" + "="*60 + "\n")
t_real, p_real, pr_real, cm_real = report(test_real, "REAL ONLY (9/class)")

=== FULL TEST (20/class) ===
alignment check (should match evaluate): 0.715
Predicted  bat  cup  drop  eat  fish  hot  jump  milk  pen  red
Actual                                                         
bat         11    1     0    1     1    0     0     0    4    2
cup          0   12     0    0     2    1     5     0    0    0
drop         0    1    17    0     0    1     1     0    0    0
eat          0    0     0   18     0    0     0     0    0    2
fish         0    1     0    0    19    0     0     0    0    0
hot          0    1     2    0     0   17     0     0    0    0
jump         0    4     0    0     0    0    14     1    0    1
milk         0    0     0    0     1    0     0    12    7    0
pen          6    0     0    0     0    0     0     0   13    1
red          0    0     0   10     0    0     0     0    0   10

              precision    recall  f1-score   support

         bat     0.6471    0.5500    0.5946        20
         cup     0.6000    0.6000    0.6000   

In [11]:
OUT = "/home/admins/lip_codebase_clean/docs/results_rebuilt_data"
os.makedirs(OUT, exist_ok=True)

cm_full.to_csv(f"{OUT}/confusion_matrix_full_test.csv")
cm_real.to_csv(f"{OUT}/confusion_matrix_real_only.csv")

for name, t, p in [("full_test", t_full, p_full), ("real_only", t_real, p_real)]:
    rep = classification_report(t, p, target_names=word_classes,
                                digits=4, output_dict=True, zero_division=0)
    pd.DataFrame(rep).transpose().to_csv(f"{OUT}/classification_report_{name}.csv")

np.save(f"{OUT}/y_true_full.npy", t_full); np.save(f"{OUT}/y_pred_full.npy", p_full)
np.save(f"{OUT}/y_prob_full.npy", pr_full)
np.save(f"{OUT}/y_true_real.npy", t_real); np.save(f"{OUT}/y_pred_real.npy", p_real)
np.save(f"{OUT}/y_prob_real.npy", pr_real)

with open(f"{OUT}/summary.txt", "w") as f:
    f.write("Model: model_rebuilt_data.h5 (2-conv, 51,434,058 params)\n")
    f.write("Dataset: rebuilt, session-based split\n")
    f.write("Training: 15 epochs, Adam 1e-4, batch 16, seed 202\n")
    f.write("Final training accuracy: 1.0000\n")
    f.write("Final validation accuracy: 0.7350\n\n")
    f.write("Full test set (20/class, 200 images):\n")
    f.write("  loss 0.8225  accuracy 0.7150  macro-F1 0.7130\n\n")
    f.write("Real recordings only (9/class, 90 images):\n")
    f.write("  loss 0.6907  accuracy 0.7556  macro-F1 0.7537\n")

print("saved")

saved


In [13]:
matplotlib.use('Agg')

OUT = "/home/admins/lip_codebase_clean/docs/results_rebuilt_data"

for cm, name, title in [
    (cm_full, "confusion_matrix_full_test",
     "Confusion Matrix — Rebuilt Data, Full Test Set (71.50%)"),
    (cm_real, "confusion_matrix_real_only",
     "Confusion Matrix — Rebuilt Data, Real Recordings Only (75.56%)")
]:
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                square=True, linewidths=0.5, linecolor='lightgray', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f"{OUT}/{name}.png", dpi=200)
    plt.close()
    print("saved:", name)

saved: confusion_matrix_full_test
saved: confusion_matrix_real_only


In [14]:
hist = pd.read_csv("/home/admins/rebuild_workspace/logs/training_history_log.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(hist.index + 1, hist['accuracy'], label='Training', marker='o', ms=3)
axes[0].plot(hist.index + 1, hist['val_accuracy'], label='Validation', marker='o', ms=3)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy — Rebuilt Data')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist.index + 1, hist['loss'], label='Training', marker='o', ms=3)
axes[1].plot(hist.index + 1, hist['val_loss'], label='Validation', marker='o', ms=3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss — Rebuilt Data')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/training_curves.png", dpi=200)
plt.close()
print("saved: training_curves")

saved: training_curves
